# Comparação LSTM vs BiLSTM (RUL FD001)

Notebook de orquestração: toda a lógica de dados, features e modelos está em `src/data.py`, `src/features.py` e `src/models.py`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "data.py").is_file():
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import (
    CYCLE_COLUMN,
    DEFAULT_RUL_LIMIT,
    ID_COLUMN,
    assign_fd001_column_names,
    compute_test_rul,
    compute_train_rul,
    drop_missing_rows,
    drop_unused_sensor_columns,
    load_fd001_raw,
)
from src.features import create_fd001_windowed_features
from src.models import (
    compare_models_wilcoxon,
    plot_rul_prediction_diagnostics,
    plot_training_history,
    predict_rul,
    run_repeated_train_eval,
    search_hyperparameters,
    split_windowed_data,
    summarize_experiment_results,
)

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
TUNER_DIR = PROJECT_ROOT / "data" / "processed" / "keras_tuner"
CHECKPOINT_LSTM = PROJECT_ROOT / "data" / "processed" / "lstm_model.h5"
CHECKPOINT_BILSTM = PROJECT_ROOT / "data" / "processed" / "bi_model.h5"
TUNER_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "data" / "processed").mkdir(parents=True, exist_ok=True)

RUL_LIMIT = DEFAULT_RUL_LIMIT
WINDOW_SIZE = 30
WINDOW_STEP = 1
N_ITER = 10

## Carga e exploração inicial

In [ ]:
raw = load_fd001_raw(DATA_RAW_DIR)
train_df, test_df = drop_missing_rows(raw.train, raw.test)
train_df, test_df, rul_df = assign_fd001_column_names(train_df, test_df, raw.rul)

print(train_df.head())

In [ ]:
fig, axes = plt.subplots(nrows=6, ncols=4, figsize=(24, 20))
axes = axes.ravel()
for i, item in enumerate(train_df.columns[2:]):
    train_df.groupby(ID_COLUMN).plot(
        kind="line", x=CYCLE_COLUMN, y=item, ax=axes[i]
    )
    axes[i].set_title(f"{item}")
    axes[i].get_legend().remove()
    axes[i].set_xlabel("Número de ciclos")
    axes[i].set_ylabel("")

plt.subplots_adjust(top=0.99, bottom=0.01, hspace=0.3, wspace=0.2)
plt.show()

## Features, targets e janelas temporais

In [ ]:
train_model, test_model = drop_unused_sensor_columns(train_df, test_df)
rul_train = compute_train_rul(train_model, limit=RUL_LIMIT)
rul_test = compute_test_rul(test_model, rul_df, limit=RUL_LIMIT)

windowed = create_fd001_windowed_features(
    train_model,
    test_model,
    rul_train,
    rul_test,
    window_size=WINDOW_SIZE,
    step=WINDOW_STEP,
)
x_train, y_train = windowed.x_train, windowed.y_train
x_test, y_test = windowed.x_test, windowed.y_test

print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

In [ ]:
exemplo = train_df[train_df[ID_COLUMN] == 1]
n_cycles = len(exemplo[CYCLE_COLUMN])
rul_motor1 = compute_train_rul(exemplo, limit=RUL_LIMIT)

plt.figure(figsize=(10, 6))
plt.plot(exemplo[CYCLE_COLUMN].iloc[:n_cycles], rul_motor1[:n_cycles], c="red")
plt.title("Representação do ciclo de vida do motor ID = 1")
plt.ylim(0, 140)
plt.xlim(0, 200)
plt.xlabel("Ciclo do motor")
plt.ylabel("RUL")
plt.show()

## Partição treino/validação interna

In [ ]:
x1, x2, y1, y2 = split_windowed_data(x_train, y_train, test_size=0.2, random_state=1)

## LSTM: busca de hiperparâmetros e avaliação repetida

In [ ]:
best_model_lstm, best_params_lstm = search_hyperparameters(
    "lstm",
    x1,
    y1,
    x2,
    y2,
    tuner_directory=TUNER_DIR,
    project_name="hyper_lstm",
    max_trials=15,
    search_epochs=5,
    overwrite=True,
)

for key, value in best_params_lstm.items():
    print(f"{key:15s} -> {value}")

In [ ]:
resultados_lstm, history_lstm = run_repeated_train_eval(
    best_model_lstm,
    x1,
    y1,
    x2,
    y2,
    x_test,
    y_test,
    CHECKPOINT_LSTM,
    n_iterations=N_ITER,
    capture_first_history=True,
)

if history_lstm is not None:
    plot_training_history(history_lstm, "LSTM")
    y_pred_lstm = predict_rul(best_model_lstm, x_test)
    plot_rul_prediction_diagnostics(
        y_test,
        y_pred_lstm,
        model_label="LSTM",
        rul_limit=RUL_LIMIT,
    )

## BiLSTM: busca de hiperparâmetros e avaliação repetida

In [ ]:
best_model_bilstm, best_params_bilstm = search_hyperparameters(
    "bilstm",
    x1,
    y1,
    x2,
    y2,
    tuner_directory=TUNER_DIR,
    project_name="hyper_bi",
    max_trials=15,
    search_epochs=5,
    overwrite=True,
)

for key, value in best_params_bilstm.items():
    print(f"{key:15s} -> {value}")

In [ ]:
resultados_bilstm, history_bilstm = run_repeated_train_eval(
    best_model_bilstm,
    x1,
    y1,
    x2,
    y2,
    x_test,
    y_test,
    CHECKPOINT_BILSTM,
    n_iterations=N_ITER,
    capture_first_history=True,
)

if history_bilstm is not None:
    plot_training_history(history_bilstm, "BiLSTM")
    y_pred_bilstm = predict_rul(best_model_bilstm, x_test)
    plot_rul_prediction_diagnostics(
        y_test,
        y_pred_bilstm,
        model_label="BiLSTM",
        rul_limit=RUL_LIMIT,
    )

## Tratamento estatístico

In [ ]:
print("Resultados LSTM".center(40, "-"))
print(f"{resultados_lstm.to_string(index=False)}\n")
print("Resultados BiLSTM".center(40, "-"))
print(f"{resultados_bilstm.to_string(index=False)}\n")

resumo_lstm = summarize_experiment_results(resultados_lstm)
resumo_bilstm = summarize_experiment_results(resultados_bilstm)

print("Resumo LSTM".center(40, "-"))
print(f"{resumo_lstm}\n")
print("Resumo BiLSTM".center(40, "-"))
print(f"{resumo_bilstm}\n")

wilcoxon_df = compare_models_wilcoxon(resultados_lstm, resultados_bilstm)
print("Teste de Wilcoxon".center(40, "-"))
for _, row in wilcoxon_df.iterrows():
    metrica = row["metrica"]
    valorp = row["p_value"]
    if row["significativo"]:
        conclusao = "A diferença é estatisticamente significativa"
    else:
        conclusao = "A diferença não é estatisticamente significativa"
    print(f"O p-value para {metrica} é {valorp}. {conclusao}")